# EXP-019 — Pre-flight only (no feature extraction)

**Purpose:** verify the experiment scripts synthetically, then reproduce the weak-label class-count audit on the provided training reports.

**Safety:** this notebook deliberately does not run full feature extraction or train a model. If any target fails the pre-registered minimum of 20 weak positives and 20 weak negatives, the EXP-019 verdict must remain **WITHHELD** under the current protocol. Do not lower that floor after seeing counts.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json

REPO_URL = 'https://github.com/vixintry/rsna-knee-m1.git'
BRANCH = 'research/exp019-weak-labels'
REPO_DIR = Path('/kaggle/working/rsna-knee-m1')
DATA_DIR = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    raise RuntimeError(f'{REPO_DIR} already exists. Remove it or use a fresh Kaggle session so the checked-out branch is unambiguous.')

assert (REPO_DIR / 'rsna-submission.ipynb').is_file(), 'Production notebook missing from clone.'
assert (REPO_DIR / 'scripts/exp019/extract_features.py').is_file(), 'EXP-019 extractor missing.'
assert (REPO_DIR / 'scripts/exp019/analyse_exp019.py').is_file(), 'EXP-019 analysis script missing.'
assert (REPO_DIR / 'scripts/derive_report_weak_labels.py').is_file(), 'Weak-label parser missing.'
assert (DATA_DIR / 'train.csv').is_file(), f'train.csv not found at {DATA_DIR}; check the competition data attachment.'
print('Repository:', REPO_DIR)
print('Branch:', BRANCH)
print('Competition train.csv:', DATA_DIR / 'train.csv')
print('Production notebook is present; it will not be edited.')

## 1. Synthetic self-tests

These test code plumbing and fail-closed gates using synthetic data. They do **not** establish model quality or permit full extraction.

In [ ]:
def run_checked(args):
    print('\n$ ' + ' '.join(map(str, args)))
    subprocess.run(args, cwd=REPO_DIR, check=True)

run_checked([sys.executable, 'scripts/exp019/extract_features.py', '--selftest'])
run_checked([sys.executable, 'scripts/exp019/analyse_exp019.py', '--selftest'])
print('\nBoth synthetic self-tests passed. Real DICOM/encoder extraction remains untested.')

## 2. Generate weak labels locally

The committed deterministic parser reads the provided `train.csv` locally. Do not send report text to a hosted third-party API. Gold-labelled studies are excluded by the parser.

In [ ]:
WEAK_DIR = Path('/kaggle/working/exp019_weak_labels')
PREFLIGHT_DIR = Path('/kaggle/working/exp019_preflight')
WEAK_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)

run_checked([
    sys.executable, 'scripts/derive_report_weak_labels.py',
    '--input-csv', str(DATA_DIR / 'train.csv'),
    '--output-dir', str(WEAK_DIR),
])
WEAK_CSV = WEAK_DIR / 'weak_labels.csv'
AUDIT_CSV = WEAK_DIR / 'weak_label_audit.csv'
assert WEAK_CSV.is_file() and AUDIT_CSV.is_file(), 'Parser did not create expected outputs.'
print('Weak labels:', WEAK_CSV)
print('Per-target cue audit:', AUDIT_CSV)

## 3. Registered class-count gate (CPU only)

Every target must have at least 20 weak positives **and** 20 weak negatives after removing gold studies. If any target fails, stop here. Do not run `extract_features.py` without first resolving the blocker under a new, pre-registered decision.

In [ ]:
audit_cmd = [
    sys.executable, 'scripts/exp019/analyse_exp019.py',
    '--audit-weak-labels',
    '--weak-labels', str(WEAK_CSV),
    '--train-csv', str(DATA_DIR / 'train.csv'),
    '--out', str(PREFLIGHT_DIR),
]
print('\n$ ' + ' '.join(map(str, audit_cmd)))
audit_run = subprocess.run(audit_cmd, cwd=REPO_DIR, text=True, capture_output=True)
print(audit_run.stdout)
if audit_run.stderr:
    print('STDERR:', audit_run.stderr)
audit_path = PREFLIGHT_DIR / 'weak_label_audit.json'
assert audit_path.is_file(), 'Expected audit JSON was not written.'
audit = json.loads(audit_path.read_text())
print('\nAll 12 targets meet the registered floor:', audit['all_12_can_be_modelled'])
print('Targets below floor:', audit['targets_below_floor'])
if not audit['all_12_can_be_modelled']:
    print('\nSTOP: EXP-019 cannot issue its registered KEEP/INVESTIGATE/REJECT verdict under the current protocol. No feature extraction or model training was run.')
else:
    print('\nPre-flight class-count gate passed only. The separate rules/quality gate is still required before any full extraction.')
print('Audit artifact:', audit_path)

## 4. Human gate still required

Do **not** set `status` to `VERIFIED` automatically. Before full extraction, a human must read the full official competition rules (including data-security terms), confirm report-derived labels are allowed under those terms, verify the metric on the official evaluation page, and complete `scripts/exp019/exp019_gates.json`. The weak-label quality gate must also be satisfied or explicitly waived in `DECISIONS.md`.

This notebook intentionally contains no full-extraction command. The production notebook and submission pipeline are not modified.